# Distributed Intelligence in the Computing Continuum

This tutorial describes a complete feedback cycle for an adaptive stream processing service that optimizes its performance throughout evolving requirements. The core phases of this cycle, following the traditional MAPE-K loop, are:

1. **Monitor**: Collect service telemetry
2. **Analyze**: Regression analysis of structural knowledge
3. **Plan**: Intent-based inference
4. **Execute**: Service scaling

Note that this abstract cycle can be implemented through various technologies and the ones presented in this tutorial present only one of many possibilities.

To reduce the experimental overhead of this tutorial, we do not actually run any physical services; instead, we operate on a prerecorded dataset that tracks the service behavior during runtime. Interested participants can find the complete experimental platform in this [repository](github.com/borissedlak/elastic-workbench).



### General Setup

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('default')

from agent.components import RASK
from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar, FIG_SIZE_DEFAULT
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

### **Monitor**: Service Telemetry

In [2]:
# TODO: Here I can just show the data structure

df_explore = pd.read_csv("../../statics/agent_experience/metrics_ICSOC_EXPLORE.csv")
df_explore_preprocessed = RASK.preprocess_data(df_explore)

INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str


### **Analyze**: Regression Analysis of Structural Knowledge

#### Create Structural Knowledge

#### Functional Data Analysis

In [3]:
# import plotly.io as pio
#
# print(pio.renderers)
# pio.renderers.default = "chrome"
# import plotly
# import jupyterlab
#
# print(plotly.__version__)
# print(jupyterlab.__version__)

In [8]:
# TODO: Make sure that one figure is rendered

# for type in ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
#          'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
#          'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
#          'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
#          'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']:

# TODO: The problem is the renderer in Pycharm. If I keep it with browser,
#  I must at least add the service and data rate to the title.
import plotly.io as pio
pio.renderers.default = 'browser'
# print(pio.renderers)

for s in services:
    _gp = GASK(s, create_figures=True, display_figures=True)
    _gp.init_model(df_explore, data_density=0.2)

# gp_list = []
# lml_history = []
#
# data_splits = 100
# tested_range = 20
#
# for i in range(tested_range):
#     gp_all_services = {}
#     data_ratio = (i + 1) / data_splits
#     lml_all_service = []
#
#     for s in services:
#         # Initialize and train GP model
#         _gp = GASK(s, create_figures=True, display_figures=False)
#         _gp.init_model(df_explore, data_density=data_ratio)
#
#         _lml = _gp.get_model_lml(s, "max_tp")
#         lml_scaled = _lml / data_ratio
#         lml_all_service.append(lml_scaled)
#         # print(f"Ratio {data_ratio}: LML = {lml:.2f}")
#         gp_all_services[s] = _gp
#
#     lml_history.append(lml_all_service)
#     gp_list.append(gp_all_services)
#
#     print(f"Finished {i + 1} / {tested_range} of the Gaussian Processes")
#
# lml_history = np.array(lml_history)


INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max_tp
/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__sigma_0 is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
INFO:multiscale:draw_3d_gp_plot took 5063 ms to execute
INFO:multiscale:train_gp_models took 5539 ms to execute
INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str
INFO:GP_Model:Fitting GP for elastic-workbench-cv-analyzer - Target: m

### Create LML Figure

In [5]:
# 1. Define the X-axis (Data Ratios)
data_ratios = np.arange(1, tested_range + 1) / data_splits

# 2. Create the Figure
plt.figure(figsize=FIG_SIZE_DEFAULT)

# Service labels matching your 'services' list order
labels = ['QR Detector', 'CV Analyzer', 'PC Visualizer']
colors = ['#d62728', '#1f77b4', '#2ca02c']  # Red, Blue, Green

for i in range(lml_history.shape[1]):
    plt.plot(data_ratios * (data_splits / tested_range), lml_history[:, i],
             marker='o', markersize=4, linewidth=2,
             label=labels[i], color=colors[i])

# 3. Formatting
abs_samples = int(len(df_explore) * (tested_range / data_splits))
# plt.title('Normalized GP Convergence (LML / samples)', fontsize=14)
plt.xlabel(f'Ratio of Training Data ({int(abs_samples / 3)} Samples)', fontsize=12)
plt.ylabel('Scaled LML / # Samples', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Service Type")

# Optimization for the display
plt.tight_layout()
plt.savefig('../figures/E1/training/lml_convergence.pdf')
RUN_HEATMAP = False

if RUN_HEATMAP:

    from agent.components.GaussianProcess import get_empirical_variable_bounds
    from utils import visualize_ndarray
    from agent.components.commons import ServiceVar
    from typing import Dict
    from agent.components.Optimizer import VersatileMapElites


    def extract_pfo_for_SLOs(gp_service, slos: Dict[ServiceVar, float], slo_type: str, simple_bounds):
        v_me = VersatileMapElites(gp_service.s_type, bins=8)

        #  I'm getting the black cells because they are not explored.
        #  What I can do is force all cells to be explored at least once,
        #  or just run gradient descent for each cell multiple (like 5) times.
        v_me.run_search(slos, gp_service, simple_bounds, iterations=2000)
        visualize_ndarray(v_me.fitness_table, gp_service.s_type.value + "_" + slo_type, cmap="viridis")

        # (2) Here I get n best solutions
        # Get n solutions that are high-performing but far apart
        # diverse_set = v_me.get_diverse_set(n_solutions=10, versatility=0.1)
        # return diverse_set
        # print("\n".join(f"Versatile Candidate: {x}" for x in diverse_set))


    final_empirical_bounds = get_empirical_variable_bounds(df_explore_preprocessed)[ServiceType.QR]
    simple_param_bounds = final_empirical_bounds.copy()
    del simple_param_bounds[ServiceVar.PERFORMANCE]
    simple_param_bounds = list(simple_param_bounds.values())

    candidate_solutions = []
    # (1) Here I give it increasingly more training data
    for i in range(tested_range):
        candidates = extract_pfo_for_SLOs(gp_list[i][ServiceType.QR], SloSet.DEFAULT.value, "DEFAULT", simple_param_bounds)
        candidate_solutions.append(candidates)


NameError: name 'tested_range' is not defined

#### Find optimal parameter configs for each SLO x Service combination after seeing certain data shares

In [ ]:
from agent.components.Optimizer import run_optimizer_multi

solution_history = []

for i in range(tested_range):
    for q in slos:
        for s in services:
            data_ratio = (i + 1) / tested_range
            gp = gp_list[i][s]

            # Run optimizer to find the best configuration
            solutions = run_optimizer_multi(s, q.value, gp, theoretical_param_bounds[s], runs=25)
            fitness, config = max(solutions, key=lambda x: x[0])

            # Predict performance (mu, sigma) for the chosen configuration
            x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
            x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
            mu, sigma = gp.predict(s, "max_tp", x_state)

            # Store everything needed for the next block
            # We include empirical_var_bounds here as it changes per iteration
            solution_history.append({
                'data_rate': data_ratio,
                'rep': None,
                'service_type': s.value,
                'slo_set': q.name,
                'p_fitness': fitness,
                # 'config': config,
                'dist': (mu, sigma),
                'cores': x_state[ServiceVar.COST],
                'data_quality': x_state[ServiceVar.QUALITY],
                'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
                # 'x_state': x_state,
            })

            print(f"Optimal fitness for {s.value} and {q.name} with {data_ratio * 100}% data: {fitness}")

#### Export to candidate solution script, with each config repeated x times

In [ ]:

repeatable_data = []
runs_per_config = 50

# Iterate through the list in steps of 3 (the size of your service triple)
for i in range(0, len(solution_history), 3):
    # Extract the current triple of rows
    triple = solution_history[i: i + 3]

    # Repeat this specific triple for the number of runs
    for run_idx in range(runs_per_config):
        for row in triple:
            new_row = row.copy()
            new_row['rep'] = run_idx + 1
            repeatable_data.append(new_row)

df_candidates = pd.DataFrame(repeatable_data)
df_candidates.to_csv(f'../statics/candidates/candidate_solutions_{tested_range}_{data_splits}_{runs_per_config}.csv',
                     index=False)
